# 13. Trade Tape -- Unified SDR Enrichment

In [3]:
import nest_asyncio
nest_asyncio.apply()

import sys, os

_nb_dir = os.path.dirname(os.path.abspath("__file__"))
_project_root = os.path.normpath(os.path.join(_nb_dir, "..", ".."))
for p in [_nb_dir, _project_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import datetime
import pandas as pd
import _usd_swaps_common as sdr

sdr.notebook_setup()

from SDRUtils.analytics.trade_tape import TradeTape

df = sdr.load_usd_swaps(
    datetime.datetime(2026, 3, 1),
    datetime.datetime(2026, 3, 10),
)
tape = TradeTape(df)
enriched = tape.compute()
print(f"Raw: {len(df):,} trades | Enriched: {len(enriched.columns)} columns")
tape.summary()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


MERGING SLICES...: 100%|██████████| 7/7 [00:00<00:00, 17.36it/s]


TradeTape:   0%|          | 0/10 [00:00<?, ?layer/s]

Raw: 32,059 trades | Enriched: 90 columns


{'n_trades': 32059,
 'n_new_risk': 32059,
 'pct_new_risk': 100.0,
 'pct_compression': 0.0,
 'pct_ufro': np.float64(41.4),
 'pct_block': np.float64(2.8),
 'pct_capped': np.float64(2.2),
 'top_trade_types': {'CURVE': 9806,
  'OUTRIGHT': 8912,
  'MATCHED_MATURITY': 3632,
  'FOMC': 3503,
  'SPREADOVER': 3434},
 'venue_split': {'D2C': 26812, 'D2D': 5247},
 'ccp_split': {'LCH': 32059}}

In [5]:
# enriched["trade_label"].value_counts()
# enriched[enriched["tape_label"].str.contains("Fed")]["tape_label"].value_counts().head(25)
# enriched["effective_date"].dt.date.value_counts()

# enriched[(enriched["effective_date"].dt.date == datetime.date(2026, 4, 29)) & (enriched["expiration_date"].dt.date == datetime.date(2026, 6, 17))][
#     "effective_date"
# ].dt.date.value_counts()

In [5]:
df["event_action"].value_counts()	

event_action
NEWT-TRAD    0
Name: count, dtype: int64

In [42]:
enriched[(enriched["tape_label"].str.lower().str.contains("vs"))]
# ["effective_date"].value_counts()

# enriched[(enriched["tape_label"] == "USD-SOFR-COMPOUND 1D Constant Spot 10Y Spreadover UFRO PHYS")]["effective_date"].value_counts()

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,days_to_quarter_end,days_to_fomc,fomc_meeting_label,fomc_proximity,execution_hour_et,execution_session,cluster_id,cluster_size,is_multi_meeting_cluster,tape_label


In [69]:
print("Top 25 tape labels by risk:")
label_risk = enriched.groupby("tape_label")["risk"].sum().sort_values(ascending=False)
for label, risk in label_risk.head(25).items():
    print(f"  ${risk/1e6:>8.0f}M  {label}")

Top 25 tape labels by risk:
  $       6M  USD-SOFR-COMPOUND 1D Constant Spot ~7Y Outright PHYS
  $       3M  USD-SOFR-OIS Compound 1D Constant Spot 10Y Outright PHYS
  $       3M  USD-SOFR-OIS Compound 1D Constant IMM_M2026 5Y/30Y CURVE UFRO PHYS
  $       3M  USD-SOFR-OIS Compound 1D Constant Spot ~7Y Outright UFRO PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot 10Y/30Y CURVE PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot 2Y Outright UFRO BLOCK PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot 15Y Outright PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot 5Y Outright PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot 5Y/10Y CURVE PHYS
  $       2M  USD-SOFR-COMPOUND 1D Constant Spot 10Y Outright PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot ~2Y Outright UFRO BLOCK PHYS
  $       2M  USD-SOFR-COMPOUND 1D Constant Spot ~7Y Outright BLOCK PHYS
  $       2M  USD-SOFR-OIS Compound 1D Constant Spot 12Y Outright PHYS
  $       2M  USD-S

In [3]:
for col in ["trade_type", "lifecycle_type", "venue", "ccp", "rate_index_clean"]:
    print(f"\n--- {col} ---")
    print(enriched[col].value_counts().to_string())


--- trade_type ---


trade_type
OUTRIGHT            8105
CURVE               7604
SPREADOVER          3715
MATCHED_MATURITY    3373
FOMC                1960
MAC                  895
IMM                  848
INVOICE_SWAP         528
FLY                  408

--- lifecycle_type ---
lifecycle_type
NEW_TRADE    27436

--- venue ---
venue
D2C    23683
D2D     3753

--- ccp ---
ccp
LCH    27436

--- rate_index_clean ---
rate_index_clean
SOFR         26975
FED_FUNDS      407
OTHER           54


In [4]:
clean = tape.clean_tape()
raw_risk = enriched["risk"].sum()
clean_risk = clean["risk"].sum()

print(f"Total Risk:  ${raw_risk/1e6:.0f}M")
print(f"Clean Risk:  ${clean_risk/1e6:.0f}M")
print(f"Noise:       ${(raw_risk-clean_risk)/1e6:.0f}M ({(1-clean_risk/raw_risk)*100:.0f}%)")
print(f"\nClean trades: {len(clean):,} / {len(enriched):,} ({len(clean)/len(enriched)*100:.0f}%)")

Total Risk:  $1034M
Clean Risk:  $560M
Noise:       $474M (46%)

Clean trades: 14,900 / 27,436 (54%)


In [5]:
from collections import Counter

all_flags = [f for flags in enriched["quality_flags"] for f in flags]
flag_counts = Counter(all_flags)
n = len(enriched)

print("Quality Flag Distribution:")
for flag, count in flag_counts.most_common():
    print(f"  {flag}: {count:,} ({count/n*100:.1f}%)")

print(f"\nUFRO: {enriched['is_ufro'].sum():,} ({enriched['is_ufro'].mean()*100:.1f}%)")
print(f"Block: {enriched['is_block'].sum():,} ({enriched['is_block'].mean()*100:.1f}%)")
print(f"Capped: {enriched['is_capped'].sum():,} ({enriched['is_capped'].mean()*100:.1f}%)")
print(f"Off-market: {enriched['is_off_market'].sum():,} ({enriched['is_off_market'].mean()*100:.1f}%)")

Quality Flag Distribution:
  UFRO: 11,674 (42.5%)
  OFF_MARKET: 7,385 (26.9%)
  CAPPED_NOTIONAL: 556 (2.0%)

UFRO: 11,674 (42.5%)
Block: 774 (2.8%)
Capped: 556 (2.0%)
Off-market: 7,385 (26.9%)


In [6]:
pkg = tape.package_summary()
print(f"Unique packages: {len(pkg):,}")
print(f"\nTop 15 package structures:")
print(pkg.groupby("package_structure")["total_risk"].agg(["count","sum"]).sort_values("sum", ascending=False).head(15).to_string())

Unique packages: 4,669

Top 15 package structures:
                      count         sum
package_structure                      
5Y/30Y Curve            118  44242500.0
7Y Matched_Maturity     460  42219200.0
10Y/30Y Curve           102  36893900.0
4Y Matched_Maturity     378  29035000.0
2Y/10Y Curve             66  28285800.0
5Y/10Y Curve             79  27870200.0
5Y Matched_Maturity     358  27835500.0
2Y Matched_Maturity     285  20371000.0
3Y Matched_Maturity     323  19266000.0
30Y Matched_Maturity    270  17451200.0
10Y Matched_Maturity    233  16487300.0
7Y Invoice_Swap         188  14424900.0
2Y/30Y Curve             28  14066000.0
2Y/5Y Curve              43  10846400.0
3Y/5Y Curve              31  10079300.0


In [7]:
session_stats = enriched.groupby("execution_session").agg(
    trades=("risk", "size"),
    total_risk=("risk", "sum"),
).sort_values("total_risk", ascending=False)
session_stats["pct"] = (session_stats["total_risk"] / session_stats["total_risk"].sum() * 100).round(1)
print(session_stats.to_string())

                   trades   total_risk   pct
execution_session                           
NY_AM               11509  436967000.0  42.3
NY_PM                7981  334839000.0  32.4
Late                 2184  110372000.0  10.7
London               3909  108784000.0  10.5
Asia                 1853   42876400.0   4.1


In [8]:
print(f"Total clusters: {enriched['cluster_id'].nunique():,}")
print(f"Multi-trade clusters: {(enriched['cluster_size'] > 1).sum():,} trades in clusters of 2+")

cluster_sizes = enriched.groupby("cluster_id").size()
print(f"\nCluster size distribution:")
print(cluster_sizes.value_counts().sort_index().head(10).to_string())

if enriched['is_multi_meeting_cluster'].any():
    n_multi = enriched['is_multi_meeting_cluster'].sum()
    print(f"\nMulti-FOMC-meeting clusters: {n_multi} trades")

Total clusters: 771
Multi-trade clusters: 27,242 trades in clusters of 2+

Cluster size distribution:
1     194
2     106
3      77
4      56
5      36
6      26
7      18
8      23
9      16
10     14

Multi-FOMC-meeting clusters: 22149 trades


In [9]:
cross = pd.crosstab(
    [enriched["trade_type"], enriched["rate_index_clean"]],
    enriched["venue"],
    values=enriched["risk"],
    aggfunc="sum",
).fillna(0)
cross = cross / 1e6  # millions
print("Risk Cross-tab (trade_type x rate_index x venue, $M):")
print(cross.round(0).to_string())

Risk Cross-tab (trade_type x rate_index x venue, $M):
venue                                D2C   D2D
trade_type       rate_index_clean             
CURVE            FED_FUNDS           0.0   0.0
                 SOFR              240.0  34.0
FLY              SOFR               15.0   0.0
FOMC             FED_FUNDS           4.0   6.0
                 SOFR               51.0   3.0
IMM              SOFR               13.0   7.0
INVOICE_SWAP     SOFR               17.0  22.0
MAC              SOFR               20.0   0.0
MATCHED_MATURITY FED_FUNDS           1.0   0.0
                 OTHER               0.0   0.0
                 SOFR              189.0  48.0
OUTRIGHT         FED_FUNDS           1.0   1.0
                 OTHER               1.0   0.0
                 SOFR              146.0  54.0
SPREADOVER       FED_FUNDS           1.0   0.0
                 SOFR              150.0  10.0
